In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Ruta de tu dataset con las características ya extraídas
ruta_dataset = '/content/drive/MyDrive/Proyecto/Daily_Sports_Activities/Data/dataset_final_features.csv'

# Cargar el dataset
df = pd.read_csv(ruta_dataset)

print(f"Dimensiones del dataset: {df.shape}")
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dimensiones del dataset: (15200, 243)


,activity,subject,window_id,T_acc_mean_x,T_acc_mean_y,T_acc_mean_z,T_acc_std_x,T_acc_std_y,T_acc_std_z,T_acc_max_x,...,LL_mag_max_x,LL_mag_max_y,LL_mag_max_z,LL_mag_corr_xy,LL_mag_corr_xz,LL_mag_corr_yz,LL_mag_mag_mean,LL_mag_mag_std,LL_mag_mag_auc,LL_mag_mag_mean_diff
0,1,1,0,8.015509,1.058076,5.553903,0.129444,0.039797,0.191729,8.1605,...,0.74182,0.30267,-0.055365,-0.380922,0.214412,-0.094971,0.800508,0.000745,2.369513,0.000941
1,1,1,1,7.920071,1.126935,5.683707,0.058170,0.026639,0.105984,8.0412,...,0.74320,0.30342,-0.054963,-0.351583,0.448888,-0.306916,0.801040,0.000701,2.371086,0.000761
2,1,1,2,8.001183,1.141395,5.559029,0.095242,0.030720,0.148445,8.1763,...,0.74335,0.30377,-0.054945,-0.231525,0.377849,-0.342104,0.801930,0.000862,2.373707,0.000829
3,1,1,3,7.941989,1.143843,5.658659,0.059360,0.024328,0.094272,8.1160,...,0.74302,0.30397,-0.054711,-0.266598,0.365792,-0.255355,0.802269,0.000731,2.374725,0.000797
4,1,1,4,7.996011,1.138048,5.567233,0.042821,0.021047,0.067826,8.0860,...,0.74316,0.30423,-0.055413,-0.169517,0.621999,-0.270246,0.802356,0.000820,2.375000,0.000938


In [ ]:
df_features = df.drop(columns=['window_id'])

In [ ]:
import time
import numpy as np
from tempfile import mkdtemp
from shutil import rmtree

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA, PCA
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier

# GRID OPTIMIZADO PARA KPCA (Pégalo en tus libretas)
GRID_KPCA = [
    {
        'reduccion__kernel': ['poly'],
        'reduccion__n_components': [32, 64],
        'reduccion__degree': [2, 3], # Solo transformaciones no lineales
        'reduccion__gamma': [None]
    }
]

def nested_loso(
    datos,
    modelo_base,         # <--- Recibe tu modelo intacto
    grid_modelo,         # <--- Recibe tu diccionario de hiperparámetros intacto
    nombre_reduccion,
    target_col="activity",
    group_col="subject",
    scoring="accuracy",
    average="weighted",
    random_state=42
):
    print(f"--- NESTED LOSO | Reducción: {nombre_reduccion.upper()} ---")

    X = datos.drop(columns=[target_col, group_col])
    y = datos[target_col]
    groups = datos[group_col]

    logo_outer = LeaveOneGroupOut()
    fold = 1

    accuracies, precisions, recalls, f1_scores = [], [], [], []
    start_time_total = time.time()

    # 1. Directorio temporal para la caché del Pipeline
    cachedir = mkdtemp()

    try:
        for train_index, test_index in logo_outer.split(X, y, groups):
            start_time_fold = time.time()

            X_test      = X.iloc[test_index]
            y_test      = y.iloc[test_index]
            sujeto_eval = groups.iloc[test_index].unique()[0]

            X_train_val      = X.iloc[train_index]
            y_train_val      = y.iloc[train_index]
            groups_train_val = groups.iloc[train_index]

            # =====================================================
            # 1. CONSTRUCCIÓN DEL PIPELINE SEGÚN REDUCCIÓN
            # =====================================================
            pasos = [('scaler', StandardScaler())]

            if nombre_reduccion == 'pca':
                pasos.append(('reduccion', PCA(n_components=0.95, random_state=random_state)))
                param_grid_red = {}

            elif nombre_reduccion == 'kpca':
                # FIX: Se añade eigen_solver='randomized' para no colapsar la memoria
                pasos.append(('reduccion', KernelPCA(random_state=random_state, n_jobs=-1, eigen_solver='randomized')))
                param_grid_red = GRID_KPCA

            elif nombre_reduccion == 'bosque':
                selector = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
                pasos.append(('reduccion', SelectFromModel(selector)))
                param_grid_red = {}

            elif nombre_reduccion == 'sin':
                param_grid_red = {}

            else:
                raise ValueError(f"Reducción '{nombre_reduccion}' no implementada.")

            # SE AÑADE TU MODELO AL PIPELINE
            pasos.append(('modelo', modelo_base))

            # FIX: Se aplica la memoria caché al pipeline
            pipeline = Pipeline(pasos, memory=cachedir)

            # =====================================================
            # 2. COMBINAR EL GRID DEL MODELO CON EL GRID DE REDUCCIÓN
            # =====================================================
            if isinstance(param_grid_red, list):
                param_grid = [{**bloque, **grid_modelo} for bloque in param_grid_red]
            else:
                param_grid = {**param_grid_red, **grid_modelo}

            # =====================================================
            # 3. INNER CV + GRID SEARCH
            # =====================================================
            inner_cv = GroupKFold(n_splits=3)

            # ¡Aquí se sigue ejecutando el GridSearchCV con tus parámetros!
            search = GridSearchCV(
                pipeline,
                param_grid,
                cv=inner_cv,
                scoring=scoring,
                n_jobs=-1
            )

            search.fit(X_train_val, y_train_val, groups=groups_train_val)

            mejor_pipeline = search.best_estimator_

            # Extraemos los parámetros limpios para la impresión
            params_limpios = {k.split('__')[-1]: v for k, v in search.best_params_.items()}

            # =====================================================
            # 4. PREDICCIÓN Y MÉTRICAS
            # =====================================================
            y_pred = mejor_pipeline.predict(X_test)

            acc_fold = accuracy_score(y_test, y_pred)
            f1_fold  = f1_score(y_test, y_pred, average=average, zero_division=0)

            accuracies.append(acc_fold)
            precisions.append(precision_score(y_test, y_pred, average=average, zero_division=0))
            recalls.append(recall_score(y_test, y_pred, average=average, zero_division=0))
            f1_scores.append(f1_fold)

            print(
                f"[Iter {fold}] Test Sujeto {sujeto_eval} -> "
                f"Params: {params_limpios} | "
                f"Acc: {acc_fold*100:.2f}% | "
                f"F1: {f1_fold*100:.2f}% | "
                f"Tiempo: {time.time() - start_time_fold:.2f} s"
            )

            fold += 1

    finally:
        # Se limpia la caché obligatoriamente al terminar
        rmtree(cachedir)

    # =========================================================
    # RESULTADOS FINALES
    # =========================================================
    print(f"\n==========================================")
    print(f"Tiempo total LOSO Anidado: {(time.time() - start_time_total) / 60:.2f} min")
    print(f"ACCURACY FINAL : {np.mean(accuracies) * 100:.2f}% (± {np.std(accuracies) * 100:.2f}%)")
    print(f"PRECISION FINAL: {np.mean(precisions) * 100:.2f}%")
    print(f"RECALL FINAL   : {np.mean(recalls) * 100:.2f}%")
    print(f"F1-SCORE FINAL : {np.mean(f1_scores) * 100:.2f}%")

    return {
        "accuracy":  np.mean(accuracies),
        "precision": np.mean(precisions),
        "recall":    np.mean(recalls),
        "f1_score":  np.mean(f1_scores),
    }

In [ ]:
from sklearn.linear_model import LogisticRegression

# 1. Defines TU modelo
mi_modelo_logreg = LogisticRegression(random_state=42, max_iter=1000)

# 2. Defines TU grid de parámetros (con el prefijo 'modelo__')
mi_grid_logreg = {
    'modelo__C': [0.01, 0.1, 1, 10],
    'modelo__penalty': ['l2'],
    'modelo__solver': ['lbfgs']
}

# 3. Llamas a la función
resultados = nested_loso(
    datos=df_features,
    modelo_base=mi_modelo_logreg,
    grid_modelo=mi_grid_logreg,
    nombre_reduccion='kpca'
)

--- NESTED LOSO | Reducción: KPCA ---
[Iter 1] Test Sujeto 1 -> Params: {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs', 'degree': 2, 'gamma': None, 'kernel': 'poly', 'n_components': 64} | Acc: 94.95% | F1: 94.28% | Tiempo: 180.74 s
[Iter 2] Test Sujeto 2 -> Params: {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs', 'degree': 3, 'gamma': None, 'kernel': 'poly', 'n_components': 64} | Acc: 91.58% | F1: 90.35% | Tiempo: 162.18 s
[Iter 3] Test Sujeto 3 -> Params: {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs', 'degree': 2, 'gamma': None, 'kernel': 'poly', 'n_components': 64} | Acc: 96.53% | F1: 96.48% | Tiempo: 138.51 s
[Iter 4] Test Sujeto 4 -> Params: {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs', 'degree': 2, 'gamma': None, 'kernel': 'poly', 'n_components': 32} | Acc: 96.42% | F1: 96.44% | Tiempo: 134.35 s
[Iter 5] Test Sujeto 5 -> Params: {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs', 'degree': 3, 'gamma': None, 'kernel': 'poly', 'n_components': 64} | Acc: 91.89% | F1: 90.01% | Tiempo: 152.26 s
[Iter

In [ ]:
nested_loso(datos=df_features, nombre_modelo='regresion', nombre_reduccion='pca')

--- NESTED LOSO | Modelo: REGRESION | Reducción: PCA ---
[Iter 1] Test Sujeto 1 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs', 'n_components': 0.9} | Tiempo: 122.76 s
[Iter 2] Test Sujeto 2 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs', 'n_components': 0.9} | Tiempo: 103.51 s
[Iter 3] Test Sujeto 3 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs', 'n_components': 0.9} | Tiempo: 98.94 s
[Iter 4] Test Sujeto 4 -> Parámetros óptimos: {'C': 0.1, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs', 'n_components': 0.99} | Tiempo: 101.20 s
[Iter 5] Test Sujeto 5 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs', 'n_components': 0.99} | Tiempo: 97.09 s
[Iter 6] Test Sujeto 6 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs', 'n_components': 0.99} | Tiempo: 93.62 s
[Iter 7] Test Sujeto 7 -> Parámetros ó

In [ ]:
nested_loso(datos=df_features, nombre_modelo='regresion', nombre_reduccion='bosque')

--- NESTED LOSO | Modelo: REGRESION | Reducción: BOSQUE ---
[Iter 1] Test Sujeto 1 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 426.35 s
[Iter 2] Test Sujeto 2 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 421.65 s
[Iter 3] Test Sujeto 3 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 417.07 s
[Iter 4] Test Sujeto 4 -> Parámetros óptimos: {'C': 0.1, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 428.33 s
[Iter 5] Test Sujeto 5 -> Parámetros óptimos: {'C': 0.1, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 426.23 s
[Iter 6] Test Sujeto 6 -> Parámetros óptimos: {'C': 0.1, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 421.67 s
[Iter 7] Test Sujeto 7 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 421.20 s
[Iter 8] Test Sujeto 8 -> Parámet

In [ ]:
nested_loso(datos=df_features, nombre_modelo='regresion', nombre_reduccion='sin')

--- NESTED LOSO | Modelo: REGRESION | Reducción: SIN ---
[Iter 1] Test Sujeto 1 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 48.48 s
[Iter 2] Test Sujeto 2 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 43.99 s
[Iter 3] Test Sujeto 3 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 44.66 s
[Iter 4] Test Sujeto 4 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 44.50 s
[Iter 5] Test Sujeto 5 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 43.28 s
[Iter 6] Test Sujeto 6 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 42.09 s
[Iter 7] Test Sujeto 7 -> Parámetros óptimos: {'C': 0.01, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'} | Tiempo: 41.10 s
[Iter 8] Test Sujeto 8 -> Parámetros ópt

In [ ]:
nested_loso(datos=df_features, nombre_modelo='regresion', nombre_reduccion='kpca')

--- NESTED LOSO | Modelo: REGRESION | Reducción: KPCA ---

[Iter 1/8] Evaluando Sujeto de Test: 1


KeyboardInterrupt: 